# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 6 – Data Analysis** (simplified, beginner version)

Pipeline position: **6 of 8**

## Objective
Answer questions about the data: how are prices spread, which features raise the price, which products are best value, top-10 lists.

## Step 1 – Setup

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


In [41]:
df = pd.read_csv("Air_Conditioners_Cleaned.csv")
print("Loaded:", df.shape)

Loaded: (652, 15)


## Step 2 – Descriptive statistics
`describe()` gives count, mean, spread (std), min, quartiles and max.

In [42]:
columns = ["discount_price", "actual_price", "discount_percent", "ratings", "no_of_ratings"]
df[columns].describe().round(2)

,discount_price,actual_price,discount_percent,ratings,no_of_ratings
count,423.00,423.00,423.00,392.00,392.00
mean,43102.61,61564.53,29.10,3.87,220.24
std,12399.14,15805.39,12.41,0.76,566.00
min,24990.00,29490.00,0.88,1.00,1.00
25%,34990.00,50795.00,21.41,3.60,3.00
50%,39990.00,59990.00,30.42,4.00,16.50
75%,47940.00,70565.00,38.98,4.20,175.75
max,128800.00,149000.00,57.26,5.00,5073.00


In [43]:
prices = df["discount_price"].dropna()
print("10% of ACs cost less than Rs", round(prices.quantile(0.10)))
print("25% of ACs cost less than Rs", round(prices.quantile(0.25)))
print("50% (median) price is   Rs", round(prices.quantile(0.50)))
print("75% of ACs cost less than Rs", round(prices.quantile(0.75)))
print("90% of ACs cost less than Rs", round(prices.quantile(0.90)))

10% of ACs cost less than Rs 31050
25% of ACs cost less than Rs 34990
50% (median) price is   Rs 39990
75% of ACs cost less than Rs 47940
90% of ACs cost less than Rs 57793


## Step 3 – Correlation
Correlation (r) is a number from **−1 to +1** that shows how two columns move together.
+1 = rise together, 0 = no link, −1 = one rises while the other falls.

In [44]:
corr_columns = ["discount_price", "actual_price", "discount_percent", "tonnage", "energy_star", "ratings", "no_of_ratings"]
corr = df[corr_columns].corr().round(2)
corr

,discount_price,actual_price,discount_percent,tonnage,energy_star,ratings,no_of_ratings
discount_price,1.00,0.79,-0.35,0.67,0.21,-0.05,-0.16
actual_price,0.79,1.00,0.28,0.57,0.24,0.07,-0.03
discount_percent,-0.35,0.28,1.00,-0.16,0.03,0.21,0.23
tonnage,0.67,0.57,-0.16,1.00,-0.05,-0.02,-0.10
energy_star,0.21,0.24,0.03,-0.05,1.00,0.11,0.13
ratings,-0.05,0.07,0.21,-0.02,0.11,1.00,0.12
no_of_ratings,-0.16,-0.03,0.23,-0.10,0.13,0.12,1.00


In [45]:
print("Price vs tonnage     : r =", round(df["discount_price"].corr(df["tonnage"]), 2))
print("Price vs energy star : r =", round(df["discount_price"].corr(df["energy_star"]), 2))
print("Price vs rating      : r =", round(df["discount_price"].corr(df["ratings"]), 2))

Price vs tonnage     : r = 0.67
Price vs energy star : r = 0.21
Price vs rating      : r = -0.05


## Step 4 – Does a higher star rating mean a higher price?

In [46]:
by_star = df.groupby("energy_star")["discount_price"].agg(["count", "mean", "median"]).round(0)
print(by_star)
print()

price_5 = df.loc[df["energy_star"] == 5, "discount_price"].mean()
price_3 = df.loc[df["energy_star"] == 3, "discount_price"].mean()
print("Average price of 5-star ACs: Rs", round(price_5))
print("Average price of 3-star ACs: Rs", round(price_3))
print("Difference                 : Rs", round(price_5 - price_3))

             count     mean   median
energy_star                         
1.0              3  47047.0  44290.0
2.0             11  38633.0  35490.0
3.0            233  40769.0  38190.0
4.0             37  42633.0  40800.0
5.0            113  46016.0  45000.0

Average price of 5-star ACs: Rs 46016
Average price of 3-star ACs: Rs 40769
Difference                 : Rs 5247


## Step 5 – Brand price compared with the overall median

In [47]:
overall_median = df["discount_price"].median()

brand = df.groupby("brand")["discount_price"].agg(["count", "mean"]).dropna()
brand["premium_vs_median_%"] = ((brand["mean"] / overall_median - 1) * 100).round(1)
brand.sort_values("mean", ascending=False).round(0)

,count,mean,premium_vs_median_%
brand,,,
Mitsubishi,5,66200.0,66.0
O General,22,58169.0,46.0
Daikin,35,47851.0,20.0
Toshiba,3,47710.0,19.0
Panasonic,23,45414.0,14.0
Voltas,60,44719.0,12.0
Electrolux,5,44190.0,10.0
LG,52,43788.0,10.0
IFB,2,42548.0,6.0


## Step 6 – Inverter and Wi-Fi: do they cost more?

In [48]:
# Let's inspect the columns to find the right ones for inverter and wi-fi
available_cols = list(df.columns)
print("Available columns in dataset:", available_cols)

# Try to find matching columns (case-insensitive or partial match)
inverter_col = [col for col in df.columns if 'inverter' in col.lower()]
wifi_col = [col for col in df.columns if 'wifi' in col.lower() or 'wi_fi' in col.lower()]

if inverter_col:
    print(f"\nGrouping by {inverter_col[0]}:")
    print(df.groupby(inverter_col[0])["discount_price"].agg(["count", "mean"]).round(0))
else:
    print("\nCould not find an inverter-related column.")

if wifi_col:
    print(f"\nGrouping by {wifi_col[0]}:")
    print(df.groupby(wifi_col[0])["discount_price"].agg(["count", "mean"]).round(0))
else:
    print("\nCould not find a Wi-Fi-related column.")

Available columns in dataset: ['name', 'main_category', 'sub_category', 'image', 'link', 'ratings', 'no_of_ratings', 'discount_price', 'actual_price', 'discount_amount', 'discount_percent', 'tonnage', 'energy_star', 'is_ac', 'brand']

Could not find an inverter-related column.

Could not find a Wi-Fi-related column.


## Step 7 – Best value: lowest price per ton

In [49]:
df["price_per_ton"] = df["discount_price"] / df["tonnage"]

cheapest = df.dropna(subset=["price_per_ton"]).sort_values("price_per_ton")
cheapest[["name", "brand", "tonnage", "energy_star", "discount_price", "price_per_ton"]].head(5).round(0)

,name,brand,tonnage,energy_star,discount_price,price_per_ton
629,Mitsubishi SRK24CW-S6 TON : 1.95 Heavy Split F...,Mitsubishi,6.0,3.0,67900.0,11317.0
24,Lloyd 1.5 Ton 3 Star Fixed Speed Window Ac (Co...,Lloyd,2.0,3.0,27440.0,18293.0
154,AmazonBasics 1.5 Ton 4 Star Fixed Speed Window...,Other,2.0,4.0,27490.0,18327.0
161,Voltas 2 Ton 2 Star Window AC (Copper 242LZH W...,Voltas,2.0,2.0,37199.0,18600.0
308,Vestar 1.5 Ton 3 Star Window AC White (Copper ...,Vestar,2.0,3.0,28000.0,18667.0


## Step 8 – Top-10 lists
A product with a 5.0 rating from just 2 customers is not as trustworthy as 4.5 from 2,000 customers. So for the *top-rated* list we only consider products with **at least 100 ratings**.

In [50]:
show = ["name", "brand", "tonnage", "energy_star", "ratings", "no_of_ratings", "discount_price", "actual_price", "discount_percent"]

most_expensive = df.sort_values("discount_price", ascending=False)[show].head(10)
top_discounted = df.dropna(subset=["discount_percent"]).sort_values("discount_percent", ascending=False)[show].head(10)

popular = df[df["no_of_ratings"] >= 100]
top_rated = popular.sort_values(["ratings", "no_of_ratings"], ascending=False)[show].head(10)

print("TOP 10 MOST EXPENSIVE")
print(most_expensive[["name", "brand", "discount_price"]].round(0))

TOP 10 MOST EXPENSIVE
                                                  name       brand  \
437  Voltas 3Ph Scroll SAC C2-NVenture I-Cassette A...      Voltas   
401  Voltas Venture i Cassette 3Ph Scroll SAC(C2-N)...      Voltas   
368     Voltas Venture Slimline Tower AC (3 Ton White)      Voltas   
626  Blue Star 2.0 Ton 3 Star Inverter Cassette SAC...   Blue Star   
268  O General 2 Ton 5 Star Inverter Split AC (Copp...   O General   
633            OGENERAL Inverter Split AC (ASGG24CGTA)   O General   
274  Mitsubishi Heavy Duty 2.2 Ton 3 Star Non Inver...  Mitsubishi   
148  MITSUBISHI HEAVY DUTY SRK25CSS-S6 2.2 Ton 3 St...  Mitsubishi   
216     Voltas Venture Slimline Tower AC (2 Ton White)      Voltas   
372  Ogeneral 2 Ton 5 Star Split Inverter AC - Whit...   O General   

     discount_price  
437        128800.0  
401        102800.0  
368         99750.0  
626         87800.0  
268         82260.0  
633         82260.0  
274         79350.0  
148         79350.0  
216      

In [51]:
print("TOP 10 MOST DISCOUNTED (only listings with a verified MRP)")
if len(top_discounted) > 0:
    print(top_discounted[["name", "brand", "discount_percent"]].round(1))
else:
    print("No listing has a verified discount in this dataset.")

TOP 10 MOST DISCOUNTED (only listings with a verified MRP)
                                                  name    brand  \
28   LG 1.0 Ton 5 Star DUAL Inverter Wi-Fi Window A...       LG   
523  iMee-1.5 Ton Inverter Split Air Conditioner wi...    Other   
5    Voltas 1.4 Ton 3 Star Inverter Split AC(Copper...   Voltas   
97   Voltas 1.5 Ton 3 Star, Inverter Split AC(Coppe...   Voltas   
470  Godrej 1.5 Ton 3 Star Inverter Split AC (AC 1....   Godrej   
153  LG 1 Ton 3 Star DUAL Inverter Split AC (Copper...       LG   
34   Voltas 0.8 Ton 3 Star, Inverter Split AC(Coppe...   Voltas   
553  MEGACOOL CENTRE-Dakin 1 Ton 3 Star Fixed Speed...    Other   
552  MEGACOOL CENTRE-Dakin 1.5 Ton 3 Star Fixed Spe...    Other   
4    Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...  Carrier   

     discount_percent  
28               57.3  
523              56.7  
5                54.9  
97               51.2  
470              50.5  
153              50.1  
34               50.0  
553        

In [52]:
print("TOP 10 BEST RATED (at least 100 ratings)")
top_rated[["name", "brand", "ratings", "no_of_ratings"]]

TOP 10 BEST RATED (at least 100 ratings)


,name,brand,ratings,no_of_ratings
111,Panasonic 1.5 Ton 4 Star Wi-Fi Twin-Cool Inver...,Panasonic,4.5,195.0
230,"LG 1.5 Ton 5 Star Inverter Split AC (Copper, S...",LG,4.4,3562.0
203,LG 1.5 Ton 5 Star Wi-Fi Inverter Split AC (Cop...,LG,4.4,769.0
200,"LG 1.5 Ton 4 Star Inverter Split AC (Copper, S...",LG,4.4,563.0
105,Panasonic 1.5 Ton 5 Star Wi-Fi Inverter Smart ...,Panasonic,4.4,118.0
12,Panasonic 1.5 Ton 5 Star Wi-Fi Inverter Smart ...,Panasonic,4.3,5073.0
23,Panasonic 1 Ton 5 Star Wi-Fi Inverter Smart Sp...,Panasonic,4.3,2178.0
7,Lloyd 1.5 Ton 5 Star Inverter Split Ac (5 In 1...,Lloyd,4.3,1494.0
128,Voltas 1.4 Ton 5 Star Inverter Adjustable Spli...,Voltas,4.3,1295.0
324,Voltas 1 Ton 3 Star Split Air Conditioner with...,Voltas,4.3,630.0


## Step 9 – Save

In [53]:
import os

most_expensive.to_csv( "Most_Expensive_Products.csv", index=False)
top_discounted.to_csv("Top_Discounted_Products.csv", index=False)
top_rated.to_csv("Top_Rated_Products.csv", index=False)

os.makedirs("aggregates", exist_ok=True)
corr.to_csv("aggregates/correlation_matrix.csv")
print("Files saved.")

Files saved.


**Output of Stage 6:** top-10 tables and the correlation matrix.  **Next → Stage 7: Data Visualization.**